# Stocks exploration

Pull SPY price history, compute indicators, and run an SMA-crossover backtest.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import matplotlib.pyplot as plt
import pandas as pd

## 1. Fetch SPY history

In [ ]:
from src.stocks.data import get_history

spy = get_history('SPY', period='2y')
print(spy.shape)
spy.tail()

## 2. Indicators

In [ ]:
from src.stocks.indicators import rsi, sma, macd

close = spy['Close']
ind = pd.DataFrame({
    'close': close,
    'sma20': sma(close, 20),
    'sma50': sma(close, 50),
    'rsi14': rsi(close, 14),
})
ind.tail()

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True, gridspec_kw={'height_ratios':[3,1]})
ind[['close','sma20','sma50']].plot(ax=ax1, title='SPY  —  close vs 20/50-day SMA')
ind['rsi14'].plot(ax=ax2, title='RSI(14)')
ax2.axhline(70, color='r', ls='--', lw=0.8)
ax2.axhline(30, color='g', ls='--', lw=0.8)
plt.tight_layout()

## 3. SMA-crossover backtest

In [ ]:
from src.stocks.backtest import run, sma_crossover_signals

signals = sma_crossover_signals(close, fast=20, slow=50)
result  = run(signals, close, fee_bps=1.0)

for k, v in result.stats.items():
    print(f'{k:>14s}: {v:.4f}')

buy_hold = (1 + close.pct_change().fillna(0)).cumprod()
pd.DataFrame({'strategy': result.equity, 'buy&hold': buy_hold}).plot(figsize=(11,4), title='Equity curve')